# Multi-Agent ReAct Evaluation: Iterative Refinement with Reasoning

**Purpose**: Run multi-agent iterative refinement system with ReAct reasoning and compute comprehensive metrics

**Output**: `outputMetrics/multiagent_react.csv` + `outputMetrics/multiagent_react_states.json`

## Multi-Agent Configuration:
- System: Multi-Agent Iterative Refinement
- Max Iterations (k): 3
- **ReAct Reasoning: ALWAYS ENABLED**
- Agents: 6 total
  - 4 LLM agents with ReAct: Generator, Evaluator, Instructions, Aggregator
  - 2 Rule-based: Entity Extractor, Entity Verifier

## Metrics Computed:
- **Cypher Similarity**: BLEU, Rouge-L, Jaro, Jaccard
- **Output Similarity**: Pass@1 Output, Jaccard Output
- **Validators**: Syntax, Schema, Properties
- **Derived Scores**: Pass@1 Score, KG Validity Score, Jaccard Output Score, JaRou Score
- **Composite**: LLMetric-Q
- **ReAct Specific**: Total reasoning steps, reasoning quality

## 1. Setup

In [ ]:
import sys
import os
import time
import csv
import json
import logging
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any
from collections import defaultdict

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / '.env')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

print(f"Notebook started: {datetime.now()}")
print(f"Working directory: {Path.cwd()}")

## 2. Import Modules

In [ ]:
# Multi-agent system
from system.orchestrator import MultiAgentOrchestrator
from system.graph_executor import GraphExecutor

# Comprehensive metrics calculator
from evaluation.comprehensive_metrics import ComprehensiveMetricsCalculator, create_metrics_dataframe

# Utilities
from utils.schema_loader import load_schema

print("Modules imported successfully")

## 3. Configuration

In [ ]:
# Experiment configuration
CONFIG = {
    "name": "multiagent_react",
    "model": os.getenv("DEFAULT_MODEL", "qwen/qwen-2.5-coder-32b-instruct"),
    "schema_type": "only_paths",
    "max_iterations": int(os.getenv("MAX_ITERATIONS", 3)),
    "temperature": float(os.getenv("TEMPERATURE", 0.0)),
    "max_tokens": 1024,  # Required for ReAct reasoning
    "rate_limit_delay": float(os.getenv("RATE_LIMIT_DELAY", 2.0)),
    "batch_size": int(os.getenv("BATCH_SIZE", 10)),
    "batch_pause": float(os.getenv("BATCH_PAUSE", 15.0)),
}

# Paths
OUTPUT_DIR = Path.cwd().parent / "outputMetrics"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILENAME = f"{CONFIG['name']}.csv"
OUTPUT_PATH = OUTPUT_DIR / OUTPUT_FILENAME

STATES_FILENAME = f"{CONFIG['name']}_states.json"
STATES_PATH = OUTPUT_DIR / STATES_FILENAME

print("Multi-Agent ReAct Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print(f"\n⚡ ReAct Reasoning: ALWAYS ENABLED")
print(f"\nOutput:")
print(f"  CSV: {OUTPUT_PATH}")
print(f"  States: {STATES_PATH}")

## 4. Load Data

In [ ]:
# Load ground truth questions
gt_file = Path.cwd().parent / "data" / "ground_truth" / "ground_truth_52.csv"

questions = []
with open(gt_file, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        questions.append({
            "id": i + 1,
            "question": row["Pertanyaan"],
            "ground_truth": row["Cypher Query"],
            "complexity": row["Tingkat Kompleksitas"],
            "reasoning_level": row["Tingkat Penalaran"],
            "sublevel": row["Sublevel"]
        })

print(f"Loaded {len(questions)} questions")

# Show distribution
from collections import Counter
print(f"\nBy Complexity: {dict(Counter(q['complexity'] for q in questions))}")
print(f"By Reasoning: {dict(Counter(q['reasoning_level'] for q in questions))}")
print(f"By Sublevel: {dict(Counter(q['sublevel'] for q in questions))}")

## 5. Initialize Systems

In [ ]:
# Load schema
schema = load_schema(CONFIG["schema_type"])
print(f"Schema loaded: {CONFIG['schema_type']} ({len(schema)} chars)")

# Initialize multi-agent orchestrator (ReAct always enabled)
orchestrator = MultiAgentOrchestrator(
    max_iterations=CONFIG['max_iterations'],
    model=CONFIG['model'],
    temperature=CONFIG['temperature'],
    max_tokens=CONFIG['max_tokens']
)

# Initialize graph executor
try:
    executor = GraphExecutor()
    print("Graph executor initialized successfully")
except Exception as e:
    print(f"Warning: Could not initialize graph executor - {e}")
    executor = None

# Initialize comprehensive metrics calculator
metrics_calculator = ComprehensiveMetricsCalculator(executor=executor)
print("Metrics calculator initialized")

print(f"\nMulti-Agent System Ready:")
print(f"  Model: {CONFIG['model']}")
print(f"  Max iterations (k): {CONFIG['max_iterations']}")
print(f"  Schema: {CONFIG['schema_type']}")
print(f"  Max tokens: {CONFIG['max_tokens']}")
print(f"  ✓ ReAct reasoning enabled in all 4 LLM agents")
print(f"  ✓ Expected token overhead: +70-80%")
print(f"  ✓ Reasoning traces will be captured")

## 6. Run Multi-Agent Inference & Evaluation

In [ ]:
from IPython.display import clear_output

# Storage for results
results = []
states = []

# Track refinement statistics
iterations_distribution = defaultdict(int)

def update_display(current, total, q_id, iterations, tokens, llmetric_q, reasoning_steps):
    """Update progress display."""
    clear_output(wait=True)
    pct = current / total * 100
    
    # Calculate running statistics
    total_tokens_so_far = sum(r.get("total_tokens", 0) for r in results)
    avg_llmetric = sum(r.get("LLMetric-Q", 0) for r in results) / current if current > 0 else 0
    avg_iterations = sum(r.get("total_iterations", 0) for r in results) / current if current > 0 else 0
    total_reasoning_steps = sum(r.get("total_reasoning_steps", 0) for r in results)
    pass_at_1_count = sum(1 for r in results if r.get("Pass@1 Output"))
    
    print(f"Progress: {current}/{total} ({pct:.1f}%)")
    print(f"Last: Q{q_id} - k={iterations}, tokens={tokens}, LLMetric-Q={llmetric_q:.1f}, reasoning={reasoning_steps}")
    print(f"")
    print(f"Running Statistics:")
    print(f"  Pass@1 Output: {pass_at_1_count}/{current} ({100*pass_at_1_count/current:.1f}%)")
    print(f"  Avg LLMetric-Q: {avg_llmetric:.2f}")
    print(f"  Avg Iterations: {avg_iterations:.2f}")
    print(f"  Total Tokens: {total_tokens_so_far:,}")
    print(f"  Total Reasoning Steps: {total_reasoning_steps}")
    
    # Show iterations distribution
    if iterations_distribution:
        print(f"\n  Iterations Distribution:")
        for k in sorted(iterations_distribution.keys()):
            count = iterations_distribution[k]
            print(f"    k={k}: {count}")

print("Starting Multi-Agent ReAct Inference & Evaluation...")
print("⚡ ReAct Reasoning: ENABLED")
print("=" * 60)
start_time = datetime.now()

for i, q in enumerate(questions):
    # Batch pause
    if i > 0 and i % CONFIG["batch_size"] == 0:
        print(f"\n[Batch pause: {CONFIG['batch_pause']}s]")
        time.sleep(CONFIG["batch_pause"])
    
    try:
        # Run multi-agent system
        state = orchestrator.run(
            question=q["question"],
            schema=schema,
            question_id=q["id"],
            ground_truth=q["ground_truth"]
        )
        
        # Extract final query
        final_query = state.final_query
        total_iterations = state.total_iterations
        total_tokens = state.total_tokens
        
        # Track iterations
        iterations_distribution[total_iterations] += 1
        
        # Extract ReAct metrics
        total_reasoning_steps = state.total_reasoning_steps
        avg_reasoning_quality = state.avg_reasoning_quality
        
        # Save state
        states.append(state.to_dict())
        
        # Compute all comprehensive metrics
        metrics = metrics_calculator.compute_all_metrics(
            question_id=q["id"],
            question=q["question"],
            ground_truth_query=q["ground_truth"],
            generated_query=final_query,
            prompt_technique="ReAct Multi-Agent",
            schema_format=CONFIG["schema_type"],
            reasoning_level=q["reasoning_level"],
            sublevel=q["sublevel"],
            complexity=q["complexity"]
        )
        
        # Add multi-agent specific metrics
        metrics["total_iterations"] = total_iterations
        metrics["total_tokens"] = total_tokens
        metrics["total_reasoning_steps"] = total_reasoning_steps
        metrics["avg_reasoning_quality"] = round(avg_reasoning_quality, 3)
        metrics["reasoning_enabled"] = True
        
        results.append(metrics)
        
        update_display(
            i + 1, len(questions), q["id"],
            total_iterations, total_tokens,
            metrics["LLMetric-Q"], total_reasoning_steps
        )
        
    except Exception as e:
        print(f"Error on Q{q['id']}: {e}")
        import traceback
        traceback.print_exc()
        
        # Create minimal error entry
        error_metrics = {
            "ID Pertanyaan": q["id"],
            "Teknik Prompt Engineering": "ReAct Multi-Agent",
            "Format Representasi Skema KG": CONFIG["schema_type"],
            "Tingkat Penalaran": q["reasoning_level"],
            "Sublevel": q["sublevel"],
            "Tingkat Kompleksitas": q["complexity"],
            "Cypher LLM": "",
            "LLMetric-Q": 0.0,
            "total_iterations": 0,
            "total_tokens": 0,
            "total_reasoning_steps": 0,
            "avg_reasoning_quality": 0.0,
            "reasoning_enabled": True
        }
        results.append(error_metrics)
    
    # Rate limiting
    if i < len(questions) - 1:
        time.sleep(CONFIG["rate_limit_delay"])

end_time = datetime.now()
duration = str(end_time - start_time)

print(f"\n\nMulti-Agent ReAct evaluation completed!")
print(f"Duration: {duration}")

## 7. Save Results

In [ ]:
import pandas as pd

# Create DataFrame with proper column ordering
df = create_metrics_dataframe(results)

# Add multi-agent specific columns at the end if not already there
if "total_iterations" not in df.columns:
    df["total_iterations"] = [r.get("total_iterations", 0) for r in results]
if "total_tokens" not in df.columns:
    df["total_tokens"] = [r.get("total_tokens", 0) for r in results]
if "total_reasoning_steps" not in df.columns:
    df["total_reasoning_steps"] = [r.get("total_reasoning_steps", 0) for r in results]
    df["avg_reasoning_quality"] = [r.get("avg_reasoning_quality", 0.0) for r in results]
    df["reasoning_enabled"] = [r.get("reasoning_enabled", True) for r in results]

# Save to CSV
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"Saved CSV: {OUTPUT_PATH}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

# Save states JSON (with reasoning traces)
with open(STATES_PATH, "w", encoding="utf-8") as f:
    json.dump(states, f, indent=2, ensure_ascii=False, default=str)
print(f"Saved States: {STATES_PATH}")
print(f"  ✓ Includes full reasoning traces for all 4 LLM agents")

## 8. Summary Statistics

In [ ]:
print("=" * 60)
print("MULTI-AGENT REACT EVALUATION SUMMARY")
print("=" * 60)

total = len(df)

print(f"\nConfiguration:")
print(f"  Model: {CONFIG['model']}")
print(f"  Max k: {CONFIG['max_iterations']}")
print(f"  Schema: {CONFIG['schema_type']}")
print(f"  ReAct: ALWAYS ENABLED")
print(f"  Max tokens: {CONFIG['max_tokens']}")

print(f"\nPerformance Metrics:")
print(f"  Pass@1 Output: {df['Pass@1 Output'].sum()}/{total} ({100*df['Pass@1 Output'].mean():.1f}%)")
print(f"  Pass@1 Score (avg): {df['Pass@1 Score'].mean():.2f}")
print(f"  KG Validity Score (avg): {df['KG Validity Score'].mean():.2f}")
print(f"  Jaccard Output Score (avg): {df['Jaccard Output Score'].mean():.2f}")
print(f"  JaRou Score (avg): {df['JaRou Score'].mean():.2f}")
print(f"  LLMetric-Q (avg): {df['LLMetric-Q'].mean():.2f}")

print(f"\nSimilarity Metrics (avg):")
print(f"  BLEU: {df['BLEU'].mean():.2f}")
print(f"  Rouge-L F1: {df['Rouge-L F1-score'].mean():.2f}")
print(f"  Jaro Similarity: {df['Jaro Similarity'].mean():.2f}")
print(f"  Jaccard Similarity: {df['Jaccard Similarity'].mean():.2f}")

print(f"\nValidator Results:")
syntax_valid = df['Syntax Validator'].sum()
schema_valid = (df['Schema Validator'] == 1.0).sum()
props_valid = ((df['Properties Validator'] == 1.0) | df['Properties Validator'].isna()).sum()
print(f"  Syntax Valid: {syntax_valid}/{total} ({100*syntax_valid/total:.1f}%)")
print(f"  Schema Valid: {schema_valid}/{total} ({100*schema_valid/total:.1f}%)")
print(f"  Properties Valid: {props_valid}/{total} ({100*props_valid/total:.1f}%)")

print(f"\nIterative Refinement:")
avg_iterations = df['total_iterations'].mean()
print(f"  Avg iterations: {avg_iterations:.2f}")
print(f"\n  Iterations Distribution:")
for k in sorted(iterations_distribution.keys()):
    count = iterations_distribution[k]
    pct = 100 * count / total
    print(f"    k={k}: {count} ({pct:.1f}%)")

print(f"\nReAct Reasoning:")
total_reasoning_steps = df['total_reasoning_steps'].sum()
avg_reasoning_steps = df['total_reasoning_steps'].mean()
avg_quality = df['avg_reasoning_quality'].mean()
print(f"  Total reasoning steps: {total_reasoning_steps}")
print(f"  Avg steps/question: {avg_reasoning_steps:.1f}")
print(f"  Avg reasoning quality: {avg_quality:.3f}")

print(f"\nCost & Performance:")
total_tokens = df['total_tokens'].sum()
print(f"  Total tokens: {total_tokens:,}")
print(f"  Avg tokens/question: {total_tokens/total:,.0f}")
print(f"  Duration: {duration}")

print(f"\nOutput:")
print(f"  CSV: {OUTPUT_PATH}")
print(f"  States: {STATES_PATH}")

print("\n" + "=" * 60)
print("✓ Complete evaluation with comprehensive metrics")
print("✓ ReAct reasoning traces saved in states JSON")
print("✓ Ready for comparison with baseline")
print("=" * 60)

## 9. Preview Results

In [ ]:
# Preview key columns
preview_cols = [
    "ID Pertanyaan", "Tingkat Kompleksitas", "Sublevel",
    "Pass@1 Output", "LLMetric-Q", "total_iterations",
    "Pass@1 Score", "KG Validity Score", "total_reasoning_steps"
]
display(df[preview_cols].head(10))